# 🎓 Modelagem Preditiva - FGV 2026

**Grupo:**
- João Victor Fortunato Teixeira Mundim (813190/2026)
- Martinho Aloisio Lenz (812869/2026)
- Nicolas Henrique Richardt da Silva (813138/2026)
- Renato Vargas Monteiro (812872/2026)
- Rodrigo Pimentel Carneiro (812440/2026)

**Professor:** Roberto Pina Rizzo, MSC, PMP, PMI-ACP, SAFe 6 Agilist

---

## 📦 1. Configuração do Ambiente

In [ ]:
# Instalar dependências
!pip install pandas numpy matplotlib seaborn scikit-learn openpyxl scipy -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import cross_val_score, LeaveOneOut
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.decomposition import PCA
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
print('✅ Bibliotecas carregadas!')

## 📥 2. Carregar Dados

In [ ]:
url = 'https://github.com/rodrigopimentel12/aula-predicao/raw/main/dados/Dados%20de%20Treinamento.xlsx'
df = pd.read_excel(url)
print(f'✅ Dados: {df.shape[0]} linhas x {df.shape[1]} colunas')
df

---
# 📝 EXERCÍCIOS DE FEATURE ENGINEERING
---

## 📌 1.1 - Avaliação das Variáveis Candidatas
**Quais variáveis parecem ser relevantes?**

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for i, col in enumerate(['Idade', 'Renda_Mensal', 'Valor_Compras', 'Numero_Compras']):
    ax = axes[i//2, i%2]
    df.boxplot(column=col, by='Classe', ax=ax)
    ax.set_title(f'{col} por Classe')
    ax.set_xlabel('Classe')
plt.suptitle('Boxplot por Classe', y=1.02)
plt.tight_layout()
plt.show()
print('💬 Idade, Renda, Valor e Número de Compras mostram clara separação entre classes.')
print('💬 Estado_Civil tem baixa variabilidade (14 Casados, 1 Solteiro) - candidata a descarte.')

## 📌 1.2 - Correlação entre Variáveis
**Há variáveis fortemente correlacionadas?**

In [ ]:
corr = df[['Idade', 'Renda_Mensal', 'Valor_Compras', 'Numero_Compras', 'Classe']].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='RdYlBu_r', center=0, fmt='.2f', square=True)
plt.title('Matriz de Correlação')
plt.show()
print('💬 Valor_Compras × Numero_Compras: 0.998 - Correlação quase perfeita!')
print('💬 Todas as variáveis numéricas têm alta correlação (>0.98)')

## 📌 1.3 - Recombinação de Variáveis

In [ ]:
df_t = df.copy()
df_t['Ticket_Medio'] = df_t['Valor_Compras'] / df_t['Numero_Compras']
print('✅ Criado: Ticket_Medio = Valor_Compras / Numero_Compras')
print(df_t.groupby('Classe')['Ticket_Medio'].mean())
print('💬 Ticket médio é similar entre classes (~150). A diferença está no volume, não no valor unitário.')

## 📌 1.4 - Descarte de Variáveis

In [ ]:
print('Estado_Civil:', df['Estado_Civil'].value_counts().to_dict())
df_t = df_t.drop(columns=['Estado_Civil'])
print('❌ Estado_Civil DESCARTADO - Apenas 1 Solteiro em 15 registros')

## 📌 1.5 - Tratamento de Valores Ausentes

In [ ]:
print('Ausentes:', df_t.isnull().sum().to_dict())
mediana = df_t['Renda_Mensal'].median()
df_t['Renda_Mensal'].fillna(mediana, inplace=True)
print(f'✅ Imputado com MEDIANA: R$ {mediana:,.2f}')
print('💬 Mediana evita viés de outliers e previne vazamento de dados.')

## 📌 1.6 - Transformação e Faixas

In [ ]:
df_t['Log_Renda'] = np.log(df_t['Renda_Mensal'])
df_t['Faixa_Idade'] = pd.cut(df_t['Idade'], bins=[0,35,50,65,100], labels=['Jovem','Adulto','Maduro','Senior'])
print('✅ Criado: Log_Renda (transformação logarítmica)')
print('✅ Criado: Faixa_Idade (binning)')
print(df_t['Faixa_Idade'].value_counts())

## 📌 1.7 - One-Hot Encoding

In [ ]:
print('Demonstração One-Hot para Estado_Civil (se mantido):')
print(pd.get_dummies(df[['Estado_Civil']], prefix='EC'))

## 📌 1.8 - Poder Discriminatório (ANOVA)

In [ ]:
print('=== TESTE ANOVA ===')
for col in ['Idade', 'Renda_Mensal', 'Valor_Compras', 'Numero_Compras']:
    g0, g1 = df_t[df_t['Classe']==0][col], df_t[df_t['Classe']==1][col]
    f, p = stats.f_oneway(g0, g1)
    sig = '✅' if p < 0.05 else '❌'
    print(f'{col}: F={f:.2f}, p={p:.4f} {sig}')

## 📌 1.9 - PCA

In [ ]:
X_pca = StandardScaler().fit_transform(df_t[['Idade','Renda_Mensal','Valor_Compras','Numero_Compras']])
pca = PCA()
pca.fit(X_pca)
print('Variância explicada:', [f'{v:.1%}' for v in pca.explained_variance_ratio_])
print(f'💬 PC1 explica {pca.explained_variance_ratio_[0]:.1%} - SIM, faz sentido reduzir para 1 dimensão!')

---
# 🤖 MODELAGEM
---

In [ ]:
X = StandardScaler().fit_transform(df_t[['Idade','Renda_Mensal','Valor_Compras','Numero_Compras']])
y = df_t['Classe']
for nome, mod in [('Logística', LogisticRegression()), ('Árvore', DecisionTreeClassifier(max_depth=3)), ('RF', RandomForestClassifier(n_estimators=50, max_depth=3))]:
    scores = cross_val_score(mod, X, y, cv=LeaveOneOut())
    print(f'{nome}: {scores.mean():.1%}')

In [ ]:
# Árvore de Decisão
tree = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X, y)
plt.figure(figsize=(16,8))
plot_tree(tree, feature_names=['Idade','Renda','Valor','Numero'], class_names=['Comum','Excelente'], filled=True, rounded=True)
plt.title('Árvore de Decisão')
plt.show()

In [ ]:
# Matriz de Confusão
mod = LogisticRegression().fit(X, y)
cm = confusion_matrix(y, mod.predict(X))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Comum','Excelente'], yticklabels=['Comum','Excelente'])
plt.title('Matriz de Confusão')
plt.xlabel('Predito'); plt.ylabel('Real')
plt.show()

In [ ]:
# Curva ROC
fpr, tpr, _ = roc_curve(y, mod.predict_proba(X)[:,1])
plt.plot(fpr, tpr, label=f'AUC={auc(fpr,tpr):.3f}')
plt.plot([0,1],[0,1],'--', color='gray')
plt.xlabel('FPR'); plt.ylabel('TPR')
plt.title('Curva ROC'); plt.legend()
plt.show()

---
**FGV 2026 - Prof. Roberto Pina Rizzo**

[GitHub](https://github.com/rodrigopimentel12/aula-predicao)